In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

failures = []

# ===== CHECK I: row count + freshness per layer =====
for layer, table in [("bronze", "orders"), ("silver", "orders"), ("gold", "fact_orders")]:
    cnt = spark.table(f"{catalog_name}.{layer}.{table}").count()
    if cnt == 0:
        failures.append(f"{layer}.{table} is empty")
    print(f"{layer}.{table}: {cnt:,} rows")

# ===== CHECK II: uniqueness of dimension keys =====
dim_keys = {
    "dim_customer": "customer_id",
    "dim_product": "product_id",
    "dim_seller": "seller_id",
    "dim_date": "date_key",
}
for dim, key in dim_keys.items():
    df = spark.table(f"{catalog_name}.{gold_schema}.{dim}")
    total = df.count()
    distinct = df.select(key).distinct().count()
    if total != distinct:
        failures.append(f"{dim}.{key} not unique: {total} rows, {distinct} distinct")
    print(f"{dim}.{key}: {'unique' if total==distinct else 'DUPLICATE'}")

In [0]:
# ===== CHECK III: referential integrity facts -> dims =====
# çdo customer_id ne fact_orders ekziston ne dim_customer?
fact = spark.table(f"{catalog_name}.{gold_schema}.fact_orders")
dim_cust = spark.table(f"{catalog_name}.{gold_schema}.dim_customer")
orphans = fact.join(dim_cust, on="customer_id", how="left_anti").count()
if orphans > 0:
    failures.append(f"fact_orders has {orphans} customer_id not in dim_customer")
print(f"fact_orders -> dim_customer: {orphans} orphans")

# çdo product_id ne fact_order_items ekziston ne dim_product?
items = spark.table(f"{catalog_name}.{gold_schema}.fact_order_items")
dim_prod = spark.table(f"{catalog_name}.{gold_schema}.dim_product")
orphans_p = items.join(dim_prod, on="product_id", how="left_anti").count()
if orphans_p > 0:
    failures.append(f"fact_order_items has {orphans_p} product_id not in dim_product")
print(f"fact_order_items -> dim_product: {orphans_p} orphans")

# ===== CHECK IV: range on price/freight/review_score =====
bad_price = items.filter((F.col("price") <= 0)).count()
bad_freight = items.filter((F.col("freight_value") < 0)).count()
bad_score = fact.filter(
    (F.col("review_score") < 1) | (F.col("review_score") > 5)
).filter(F.col("review_score").isNotNull()).count()

if bad_price > 0: failures.append(f"{bad_price} items with price <= 0")
if bad_freight > 0: failures.append(f"{bad_freight} items with freight < 0")
if bad_score > 0: failures.append(f"{bad_score} reviews with score out of 1-5")
print(f"Range: price<=0:{bad_price}, freight<0:{bad_freight}, score out:{bad_score}")

In [0]:
# GATE: nese ndonje check deshtoi, NDAL job-in
if failures:
    print("\n=== DATA QUALITY FAILURES ===")
    for f in failures:
        print(f"  ✗ {f}")
    raise Exception(f"Data quality gate failed: {len(failures)} issue(s)")
else:
    print("\n✓ All data quality checks passed")